In [1]:
from pathlib import Path

import pandas as pd
from statsmodels.stats.proportion import (
    proportions_ztest,
    confint_proportions_2indep
)

# Load A/B test summary created in SQL
BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "ab_test_summary.csv"

data = pd.read_csv(DATA_PATH)

data
home = data[data["landing_page"] == "/home"].iloc[0]
lander = data[data["landing_page"] == "/lander-1"].iloc[0]

home_rate = home["bounces"] / home["total_sessions"]
lander_rate = lander["bounces"] / lander["total_sessions"]

difference = lander_rate - home_rate

print(f"Home bounce rate: {home_rate:.2%}")
print(f"Lander bounce rate: {lander_rate:.2%}")
print(f"Difference: {difference * 100:.2f} pp")
# Test whether the observed bounce rates differ between landing pages
count = [
    lander["bounces"],
    home["bounces"]
]

nobs = [
    lander["total_sessions"],
    home["total_sessions"]
]

z_stat, p_value = proportions_ztest(count, nobs)

print(f"Z-statistic: {z_stat:.3f}")
print(f"P-value: {p_value:.3f}")
# 95% CI for the difference: /lander-1 minus /home
ci_low, ci_high = confint_proportions_2indep(
    count1=lander["bounces"],
    nobs1=lander["total_sessions"],
    count2=home["bounces"],
    nobs2=home["total_sessions"],
    compare="diff",
    alpha=0.05
)

print(
    f"95% CI for difference: "
    f"{ci_low * 100:.2f} to {ci_high * 100:.2f} pp"
)
ab_results = pd.DataFrame({
    "Metric": [
        "Home bounce rate",
        "Lander bounce rate",
        "Difference (Lander - Home)",
        "95% CI for difference",
        "Z-statistic",
        "P-value"
    ],
    "Result": [
        f"{home_rate * 100:.2f}%",
        f"{lander_rate * 100:.2f}%",
        f"{difference * 100:.2f} pp",
        f"{ci_low * 100:.2f} to {ci_high * 100:.2f} pp",
        f"{z_stat:.2f}",
        f"{p_value:.3f}"
    ]
})

ab_results

Home bounce rate: 53.50%
Lander bounce rate: 53.00%
Difference: -0.50 pp
Z-statistic: -0.368
P-value: 0.713
95% CI for difference: -3.17 to 2.16 pp


,Metric,Result
0,Home bounce rate,53.50%
1,Lander bounce rate,53.00%
2,Difference (Lander - Home),-0.50 pp
3,95% CI for difference,-3.17 to 2.16 pp
4,Z-statistic,-0.37
5,P-value,0.713
